# Voxel Field Extraction: CPU vs CUDA Performance Comparison

This notebook here compares the performance of the original CPU implementation vs the CUDA implementation of the voxel field extraction algorithm.

In [1]:
import torch
import numpy as np
import time
import matplotlib.pyplot as plt
import os
import sys

sys.path.append('.')

from compute_voxel_field_wrapper import (
    extract_fields_cpu, 
    extract_fields_cuda, 
    prepare_gaussians_for_extraction
)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Using device: {device}")

PyTorch version: 2.6.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU
Using device: cuda


## Generate Random Test Data

In [2]:
def generate_test_gaussians(n_gaussians=1000, device='cuda'):
    """
    Generate random Gaussian parameters for testing.
    """
    torch.manual_seed(42)
    
    means = torch.randn(n_gaussians, 3, device=device) * 2.0
    
    log_scales = torch.randn(n_gaussians, 3, device=device) * 0.5 - 2.0
    scales = torch.exp(log_scales)
    
    quaternions = torch.randn(n_gaussians, 4, device=device)
    quaternions = quaternions / quaternions.norm(dim=1, keepdim=True)
    
    opacity_logits = torch.randn(n_gaussians, 1, device=device) * 2.0
    opacities = torch.sigmoid(opacity_logits)
    
    return means, scales, quaternions, opacities

n_gaussians = 5000
means, scales, quaternions, opacities = generate_test_gaussians(n_gaussians, device)
print(f"Generated {n_gaussians} test Gaussians")
print(f"Means shape: {means.shape}")
print(f"Scales shape: {scales.shape}")
print(f"Quaternions shape: {quaternions.shape}")
print(f"Opacities shape: {opacities.shape}")

Generated 5000 test Gaussians
Means shape: torch.Size([5000, 3])
Scales shape: torch.Size([5000, 3])
Quaternions shape: torch.Size([5000, 4])
Opacities shape: torch.Size([5000, 1])


## Prepare Gaussians for Extraction

In [3]:
normalized_means, covs, filtered_opacities, scale, center = prepare_gaussians_for_extraction(
    means, scales, quaternions, opacities.squeeze(-1)
)

print(f"After filtering: {normalized_means.shape[0]} Gaussians")
print(f"Scale factor: {scale:.4f}")
print(f"Center: {center}")
print(f"Normalized means range: [{normalized_means.min().item():.3f}, {normalized_means.max().item():.3f}]")
print(f"Covariances shape: {covs.shape}")

After filtering: 4961 Gaussians
Scale factor: 0.1200
Center: tensor([ 0.3617, -0.0552, -0.4838], device='cuda:0')
Normalized means range: [-0.900, 0.900]
Covariances shape: torch.Size([4961, 6])


## Check CUDA Setup

In [4]:
import os
import ctypes
import platform
import sys

print("System Information:")
print(f"Platform: {platform.system()}")
print(f"Python version: {platform.python_version()}")
print(f"Current working directory: {os.getcwd()}")

if hasattr(sys.modules['__main__'], '__file__'):
    script_dir = os.path.dirname(os.path.abspath(sys.modules['__main__'].__file__))
else:
    script_dir = os.getcwd()
print(f"Working directory: {script_dir}")

dll_name = "compute_voxel_field.dll" if platform.system() == "Windows" else "compute_voxel_field.so"
dll_paths = [
    os.path.join(os.getcwd(), dll_name),
    os.path.join(script_dir, dll_name),
    dll_name
]

print(f"\nSearching for {dll_name}:")
found_dll = False
for path in dll_paths:
    abs_path = os.path.abspath(path)
    exists = os.path.exists(path)
    print(f"  {abs_path}: {'FOUND' if exists else 'NOT FOUND'}")
    if exists:
        found_dll = True
        print(f"    Size: {os.path.getsize(path)} bytes")
        print(f"    Full path: {os.path.abspath(path)}")

if not found_dll:
    print(f"\nError:  {dll_name} not found in any expected location!")
    print("Please make sure you compiled it in the current directory:")
    print(f"  cd {os.getcwd()}")
    print(f"  nvcc -O3 -shared -o {dll_name} compute_voxel_field.cu")

print(f"\nFiles in current directory ({os.getcwd()}):")
files = os.listdir('.')
dll_files = [f for f in files if f.endswith('.dll') or f.endswith('.so')]
cu_files = [f for f in files if f.endswith('.cu')]
py_files = [f for f in files if f.endswith('.py')]

if dll_files:
    print("  DLL/SO files:")
    for f in dll_files:
        print(f"    - {f} ({os.path.getsize(f)} bytes)")
else:
    print("  No DLL/SO files found")

if cu_files:
    print("  CUDA source files:")
    for f in cu_files:
        print(f"    - {f}")
        
if py_files:
    print("  Python files:")
    for f in py_files[:5]:  
        print(f"    - {f}")

print("\nAttempting to load CUDA library...")
try:
    wrapper_path = os.path.join(script_dir, "compute_voxel_field_wrapper.py")
    if not os.path.exists(wrapper_path):
        print(f"⚠️  Wrapper file not found at: {wrapper_path}")
        print("Make sure compute_voxel_field_wrapper.py is in the current directory")
    else:
        print(f"✓ Found wrapper at: {wrapper_path}")
    
    from compute_voxel_field_wrapper import load_cuda_library
    lib = load_cuda_library()
    print("✓ CUDA library loaded successfully!")
except Exception as e:
    print(f"✗ Failed to load CUDA library: {e}")
    
    if platform.system() == "Windows":
        print("\nWindows-specific debugging:")
        print("Make sure you have:")
        print("1. CUDA toolkit installed")
        print("2. Visual Studio C++ redistributables installed")
        print("3. CUDA runtime DLLs in your PATH")
        
        cuda_path = os.environ.get('CUDA_PATH', 'Not found')
        print(f"\nCUDA_PATH environment variable: {cuda_path}")
        
        if cuda_path != 'Not found' and os.path.exists(cuda_path):
            cuda_bin = os.path.join(cuda_path, 'bin')
            if os.path.exists(cuda_bin):
                print(f"CUDA bin directory exists: {cuda_bin}")
                important_dlls = ['cudart64_*.dll', 'cublas64_*.dll', 'cufft64_*.dll']
                print("Checking for CUDA runtime DLLs:")
                for dll_pattern in important_dlls:
                    import glob
                    matches = glob.glob(os.path.join(cuda_bin, dll_pattern))
                    if matches:
                        print(f"  ✓ Found {dll_pattern}: {os.path.basename(matches[0])}")
                    else:
                        print(f"  ✗ Missing {dll_pattern}")
        
        path_dirs = os.environ.get('PATH', '').split(';')
        cuda_in_path = [p for p in path_dirs if 'cuda' in p.lower()]
        print(f"\nCUDA directories in PATH:")
        if cuda_in_path:
            for p in cuda_in_path:
                print(f"  - {p}")
        else:
            print("  No CUDA directories found in PATH!")
            print("  You may need to add CUDA\\v11.x\\bin to your PATH")

if platform.system() == "Windows" and found_dll:
    print("\nTrying alternative DLL loading methods...")
    dll_full_path = None
    for path in dll_paths:
        if os.path.exists(path):
            dll_full_path = os.path.abspath(path)
            break
    
    if dll_full_path:
        print(f"Attempting to load: {dll_full_path}")
        try:
            lib1 = ctypes.CDLL(dll_full_path)
            print("✓ Method 1 (CDLL) successful")
        except Exception as e:
            print(f"✗ Method 1 failed: {e}")
            
        try:
            lib2 = ctypes.WinDLL(dll_full_path)
            print("✓ Method 2 (WinDLL) successful")
        except Exception as e:
            print(f"✗ Method 2 failed: {e}")
            
        try:
            lib3 = ctypes.CDLL(dll_full_path, winmode=0)
            print("✓ Method 3 (CDLL with winmode=0) successful")
        except Exception as e:
            print(f"✗ Method 3 failed: {e}")

System Information:
Platform: Windows
Python version: 3.10.16
Current working directory: c:\Users\basel\Desktop\dream gauss\final\Word2World
Working directory: c:\Users\basel\Desktop\dream gauss\final\Word2World

Searching for compute_voxel_field.dll:
  c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll: FOUND
    Size: 180224 bytes
    Full path: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll
  c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll: FOUND
    Size: 180224 bytes
    Full path: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll
  c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll: FOUND
    Size: 180224 bytes
    Full path: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll

Files in current directory (c:\Users\basel\Desktop\dream gauss\final\Word2World):
  DLL/SO files:
    - compute_voxel_field.dll (180224 bytes)
  CUDA 

## Check DLL Exports

In [5]:
import subprocess
import os

dll_path = os.path.join(os.getcwd(), "compute_voxel_field.dll")

if os.path.exists(dll_path):
    print(f"Checking exports from: {dll_path}\n")
    
    try:
        result = subprocess.run(
            ["dumpbin", "/EXPORTS", dll_path], 
            capture_output=True, 
            text=True, 
            shell=True
        )
        if result.returncode == 0:
            print("Exported functions (via dumpbin):")
            lines = result.stdout.split('\n')
            in_exports = False
            for line in lines:
                if 'ordinal hint' in line.lower():
                    in_exports = True
                elif in_exports and line.strip() and not line.startswith(' '):
                    parts = line.split()
                    if len(parts) >= 4:
                        print(f"  - {parts[-1]}")
        else:
            print("dumpbin not available (install Visual Studio developer tools)")
    except:
        print("Could not run dumpbin")
    
    print("\nTrying to load specific function with ctypes:")
    try:
        import ctypes
        lib = ctypes.CDLL(dll_path)
        
        try:
            func = lib.compute_voxel_field_cuda
            print("✓ Successfully found compute_voxel_field_cuda function!")
        except AttributeError:
            print("✗ Function compute_voxel_field_cuda not found")
            
            print("\nAvailable attributes in the library:")
            for attr in dir(lib):
                if not attr.startswith('_'):
                    print(f"  - {attr}")
                    
    except Exception as e:
        print(f"Error loading DLL: {e}")

Checking exports from: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll

dumpbin not available (install Visual Studio developer tools)

Trying to load specific function with ctypes:
✓ Successfully found compute_voxel_field_cuda function!


## Test Different Resolutions

In [6]:
resolutions = [64, 128]
num_blocks = 16
relax_ratio = 1.5

results = {}

for resolution in resolutions:
    print(f"\n{'='*50}")
    print(f"Testing resolution: {resolution}³")
    print(f"{'='*50}")
    
    print("Warming up CPU...")
    _ = extract_fields_cpu(normalized_means, covs, filtered_opacities.squeeze(-1), 
                          resolution=resolution, num_blocks=num_blocks, relax_ratio=relax_ratio)
    
    print("\nRunning CPU implementation...")
    if device == 'cuda':
        torch.cuda.synchronize()
    
    start_time = time.time()
    occ_cpu = extract_fields_cpu(
        normalized_means, covs, filtered_opacities.squeeze(-1),
        resolution=resolution, num_blocks=num_blocks, relax_ratio=relax_ratio
    )
    
    if device == 'cuda':
        torch.cuda.synchronize()
    cpu_time = time.time() - start_time
    print(f"CPU time: {cpu_time:.3f} seconds")
    
    try:
        print("\nRunning CUDA implementation...")
        _ = extract_fields_cuda(
            normalized_means, covs, filtered_opacities.squeeze(-1),
            resolution=resolution, num_blocks=num_blocks, relax_ratio=relax_ratio
        )
        
        if device == 'cuda':
            torch.cuda.synchronize()
        
        start_time = time.time()
        occ_cuda = extract_fields_cuda(
            normalized_means, covs, filtered_opacities.squeeze(-1),
            resolution=resolution, num_blocks=num_blocks, relax_ratio=relax_ratio
        )
        
        if device == 'cuda':
            torch.cuda.synchronize()
        cuda_time = time.time() - start_time
        print(f"CUDA time: {cuda_time:.3f} seconds")
        
        diff = torch.abs(occ_cpu - occ_cuda)
        max_diff = diff.max().item()
        mean_diff = diff.mean().item()
        
        nonzero_mask = occ_cpu.abs() > 1e-8
        if nonzero_mask.any():
            rel_error = (diff[nonzero_mask] / occ_cpu[nonzero_mask].abs()).mean().item()
        else:
            rel_error = 0.0
        
        print(f"\nAccuracy comparison:")
        print(f"  Max absolute difference: {max_diff:.2e}")
        print(f"  Mean absolute difference: {mean_diff:.2e}")
        print(f"  Mean relative error: {rel_error:.2e}")
        
        speedup = cpu_time / cuda_time
        print(f"\nSpeedup: {speedup:.2f}x")
        
        results[resolution] = {
            'cpu_time': cpu_time,
            'cuda_time': cuda_time,
            'speedup': speedup,
            'max_diff': max_diff,
            'mean_diff': mean_diff,
            'rel_error': rel_error,
            'occ_cpu': occ_cpu,
            'occ_cuda': occ_cuda
        }
        
    except Exception as e:
        print(f"\nCUDA implementation failed: {e}")
        print("Make sure the CUDA library is compiled correctly.")
        import traceback
        traceback.print_exc()


Testing resolution: 64³
Warming up CPU...

Running CPU implementation...
CPU time: 4.623 seconds

Running CUDA implementation...
Successfully loaded CUDA library from: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll
Successfully loaded CUDA library from: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll
CUDA time: 0.017 seconds

Accuracy comparison:
  Max absolute difference: 2.59e+00
  Mean absolute difference: 2.94e-02
  Mean relative error: 3.01e+04

Speedup: 277.57x

Testing resolution: 128³
Warming up CPU...

Running CPU implementation...
CPU time: 4.391 seconds

Running CUDA implementation...
Successfully loaded CUDA library from: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll
Successfully loaded CUDA library from: c:\Users\basel\Desktop\dream gauss\final\Word2World\compute_voxel_field.dll
CUDA time: 0.098 seconds

Accuracy comparison:
  Max absolute difference: 2.77e+00
  Mean absolute difference